🔰PyTorchでニューラルネットワーク基礎 #32 【MLM事前学習編・転移学習】

### 内容
* Qiitaの記事と連動しています
* 第31回の事前学習モデルを利用したテキスト分類
* 利用するデータ：history_text_label_id.jsonl
* 各種ファイルの保存先は環境によって適宜変更してください


### データについて
* wikipediaの日本の歴史分野の内容から時代区分を判定するテキスト分類問題
* 時代区分を直接表すテキストは除いてある。（江戸時代ラベルの文章に「江戸」という単語がはありません）
* 詳細は記事の内容を確認
* クラス数：５（弥生、奈良、室町、江戸、昭和）
* 各クラス200個の文 (合計1000個)
* idsの長さ（系列長・文章の長さ）：64

### トークナイザーについて
* 第31回の事前学習で利用したトークナイザーを使います。
* unigram_tokenizer_2k.json：unigram lmで学習した2000トークン

### 事前学習モデル
* unigram_2k.model：第31回で学習したモデルであればOK

**Transformer Encoderによるテキスト分類の基本構造**
> 埋め込み層→TransformerEncoder層→\<bos\>へ集約→FC

**【検証精度】**
* 事前学習あり：80%前後
* 事前学習なし：60%前後

In [1]:
# ライブラリの読み込み
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from sklearn.model_selection import train_test_split

#デバイスの選択
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("利用デバイス:", device.type)

# 精度を計算する関数
def accuracy(y, t):
    _, argmax_list = torch.max(y, dim=1)
    accuracy = sum(argmax_list == t).item()/len(t)
    return accuracy

利用デバイス: cuda


In [2]:
# 事前学習で保存したモデル名
pre_trained_model = "./model/unigram_2k.model"

# 保存したトークナイザーで確認
tokenizer_filename = "tokenizer/unigram_tokenizer_2k.json"
tokenizer = Tokenizer.from_file(tokenizer_filename)

In [ ]:
# (1) データファイルの読み込みJSONファイルを読み込む
filename = "./data/history_text_label_id.jsonl"
df = pd.read_json(filename, lines=True)


# (2) 系列長を等長にする
# x: IDベクトル
# t: ラベル
target_length = 64
df["ids"] = df["ids"].apply(lambda x: x + [0] * (target_length - len(x))) # 系列長が異なるので注意！64 になるように0で埋める
x = torch.LongTensor(df["ids"])
t = torch.LongTensor(df["label"])

# 学習データと訓練データに分割
x, x_test, t, t_test = train_test_split(x,t, stratify=t,  random_state=55)
x = x.to(device)
t = t.to(device)
x_test = x_test.to(device)
t_test = t_test.to(device)

### ネットワークモデルの定義と作成
1. 事前学習時に用いたmlm_headブロックをテキスト分類用のLinearに変更
2. テキスト分類用Linearへの入力には<bos>部分の特徴量を利用

In [4]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.d_model = 64
        self.seq_len = 64
        self.nhead = 4
        self.dim_feedforward = 256
        self.num_layers = 6
        self.dropout = 0.1
        self.out_features = 5
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.mask_token_id = tokenizer.token_to_id("<mask>")
        self.bos_token_id = tokenizer.token_to_id("<bos>")
        self.eos_token_id = tokenizer.token_to_id("<eos>")
        self.unk_token_id = tokenizer.token_to_id("<unk>")

        # 特殊トークンのセット
        self.special_tokens = {
            self.pad_token_id,
            self.mask_token_id,
            self.bos_token_id,
            self.eos_token_id,
            self.unk_token_id,
        }
        
        # 通常トークンのリスト special_tokenを除くトークンのリスト（MLMランダム置換用）
        self.normal_tokens = [
            i for i in range(self.vocab_size) 
            if i not in self.special_tokens
        ]
        
        # 学習設定 (今回は利用しないけど使うと便利かも)
        self.batch_size = 32
        self.learning_rate = 0.0001
        self.num_epochs = 100
        self.mask_prob = 0.15
        self.max_grad_norm = 1.0


class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        # 埋め込み層
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size, 
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id
        )
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # MLM用の出力層：BERT風レイヤー　各トークン位置で語彙全体を予測
        self.classifier = nn.Linear(in_features=config.d_model, out_features=config.out_features)

    
    def forward(self, x):
        # マスクの作成
        src_key_padding_mask = (x == self.config.pad_token_id)
        
        # 埋め込み
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(x.size(1), device=x.device))
        x = tok_emb + pos_emb.unsqueeze(0)
        
        x = self.layer_norm(x)
        x = self.dropout(x)
        
        # Transformer Encoder
        h = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)

        # ---- 文ベクトルへの Pooling  <BOS>トークン（先頭）に情報を集約
        pooled = h[:, 0, :]  # [batch, d_model]
        
        # ---- 分類 ----
        y = self.classifier(pooled)  # [batch, num_labels=5]    
        return y

## 学習済みモデルの読み込み
* torch.load(事前学習モデルファイル)で学習したモデルの重みや設定を読み込む
* tokenizerを指定して、configを更新
* mlm_head: 事前学習のネットワークモデルでつけた名前
* mlm_headを除いて model.load_state_dict()することで、mlm_head以外の事前学習の重みを読み込める

In [5]:
# (3) 事前学習したモデルファイルを読み込む
checkpoint = torch.load(pre_trained_model, map_location=device)

config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])

# (4) configを元にDNNを作成（この段階だとまだ事前学習と同じネットワーク構造）
model = DNN(config).to(device)


# (5) 事前学習重み（checkpoint側）をフィルタ
pretrained = checkpoint["model_state_dict"]
pretrained_filtered = {
    k: v for k, v in pretrained.items()
    if (not k.startswith("mlm_head."))
}

# (6) 必要な部分（事前学習モデルの一部分）の重みだけ必要なのでstrict=False にする。
info = model.load_state_dict(pretrained_filtered, strict=False)
print(f"削除されているか確認: {info.unexpected_keys}") # 何も表示されないはず


削除されているか確認: []


### 転移学習
* 更新するパラメータを決める

今回更新するのは、
* 分類に利用する全結合層のパラメータ
* Transformerの最終層から2層（ここ変更すると色々変わります）

となります。

In [6]:
params_to_update = []

for name , param in model.named_parameters():
    param.requires_grad = False
for name, param in model.transformer_encoder.layers[-2:].named_parameters():  # 最終層+1
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name)
for name, param in model.classifier.named_parameters():
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name) 


更新されるパラメータ: 0.self_attn.in_proj_weight
更新されるパラメータ: 0.self_attn.in_proj_bias
更新されるパラメータ: 0.self_attn.out_proj.weight
更新されるパラメータ: 0.self_attn.out_proj.bias
更新されるパラメータ: 0.linear1.weight
更新されるパラメータ: 0.linear1.bias
更新されるパラメータ: 0.linear2.weight
更新されるパラメータ: 0.linear2.bias
更新されるパラメータ: 0.norm1.weight
更新されるパラメータ: 0.norm1.bias
更新されるパラメータ: 0.norm2.weight
更新されるパラメータ: 0.norm2.bias
更新されるパラメータ: 1.self_attn.in_proj_weight
更新されるパラメータ: 1.self_attn.in_proj_bias
更新されるパラメータ: 1.self_attn.out_proj.weight
更新されるパラメータ: 1.self_attn.out_proj.bias
更新されるパラメータ: 1.linear1.weight
更新されるパラメータ: 1.linear1.bias
更新されるパラメータ: 1.linear2.weight
更新されるパラメータ: 1.linear2.bias
更新されるパラメータ: 1.norm1.weight
更新されるパラメータ: 1.norm1.bias
更新されるパラメータ: 1.norm2.weight
更新されるパラメータ: 1.norm2.bias
更新されるパラメータ: weight
更新されるパラメータ: bias


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params_to_update, lr=1e-3)

In [8]:
LOOP = 100
model.train()
for epoch in range(LOOP):
    optimizer.zero_grad()

    y = model(x)
    loss = criterion(y, t)
    acc  = accuracy(y,t)
    loss.backward()
    optimizer.step()

    if (epoch+1)%10==0:  
        print(f"{epoch}:\tloss:{loss.item():.3f}\tacc:{acc:.3f}")


9:	loss:1.553	acc:0.621
19:	loss:1.418	acc:0.720
29:	loss:1.277	acc:0.757
39:	loss:1.142	acc:0.823
49:	loss:1.011	acc:0.839
59:	loss:0.860	acc:0.896
69:	loss:0.732	acc:0.912
79:	loss:0.591	acc:0.943
89:	loss:0.476	acc:0.956
99:	loss:0.384	acc:0.957


In [9]:
model.eval()
with torch.inference_mode():
    y_test = model(x_test)
    acc = accuracy(y_test, t_test)
print(f"検証精度: {acc}")

検証精度: 0.8
